In [ ]:
%pip install anthropic python-dotenv

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [35]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"

In [ ]:
def append_user_message(messages, text):
    msg = {"role":"user","content":text}
    messages.append(msg)

def append_assistant_message(messages, text):
    msg = {"role":"assistant","content":text}
    messages.append(msg)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages
    )
    return message.content[0].text

In [ ]:
conversation = []

append_user_message(conversation, "how do i solve 5x + 3 = 2 for x?")

answer = chat(conversation)

/var/folders/nm/31xhm9yj71lfzvy5jfq2gc640000gn/T/ipykernel_66904/2555223964.py:10: DeprecationWarning: The model 'claude-sonnet-4-0' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  message = client.messages.create(


In [6]:
print(answer)

To solve 5x + 3 = 2 for x, I need to isolate x by using inverse operations.

**Step 1:** Subtract 3 from both sides
5x + 3 - 3 = 2 - 3
5x = -1

**Step 2:** Divide both sides by 5
5x ÷ 5 = -1 ÷ 5
x = -1/5

**Check:** Let me verify by substituting back into the original equation:
5(-1/5) + 3 = -1 + 3 = 2 ✓

Therefore, **x = -1/5** (or -0.2 in decimal form).


In [32]:
def chat2(messages, system=None, stop_sequences=None):
    params = {
        "max_tokens":1000,
        "model":model,
        "messages":messages
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
# system promts 
# System prompts are a powerful way to customize how Claude responds to user input. 
# Instead of getting generic answers, you can shape Claude's tone, style, and approach to match your specific use case.
system="""
    you are a patient math tutor. do not directly answer a student's questions.
    guide them to a solution step by step
    """
conversation2 = []
append_user_message(conversation2, "how do i solve 5x + 3 = 2 for x?")
answer = chat2(conversation2, system=system)
print(answer)

In [21]:
conversation = []
append_user_message(conversation, "write a golang function that checks a string for duplication characters")
answer = chat2(conversation, system="you are a senior golang engineer who writes optimized, concise code")
print(answer)
print("============================== END ================================")

system="""
Do not provide direct code examples, provide only the hints. probably you can provide 60% of the solution
"""
answer = chat2(conversation, system=system)
print(answer)


Here's a clean, optimized solution with multiple approaches:

```go
package main

import "fmt"

// HasDuplicateChars checks if a string contains duplicate characters.
// Uses a bitmask for ASCII or a map for full Unicode support.

// For ASCII strings (fast, O(n) time, O(1) space)
func HasDuplicateCharsASCII(s string) bool {
	var seen [256]bool
	for i := 0; i < len(s); i++ {
		if seen[s[i]] {
			return true
		}
		seen[s[i]] = true
	}
	return false
}

// For Unicode strings (O(n) time, O(n) space)
func HasDuplicateChars(s string) bool {
	seen := make(map[rune]struct{})
	for _, r := range s {
		if _, exists := seen[r]; exists {
			return true
		}
		seen[r] = struct{}{}
	}
	return false
}

// GetDuplicateChars returns all duplicate characters found in the string
func GetDuplicateChars(s string) []rune {
	count := make(map[rune]int)
	for _, r := range s {
		count[r]++
	}

	var duplicates []rune
	for r, c := range count {
		if c > 1 {
			duplicates = append(duplicates, r)
		}
	}
	return dup

In [25]:
messages = []
append_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_01D8wW53wdgnN23ZYM59zKnj', container=None, content=[], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=18, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='Here', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' is a one sentence description of a fake database:\n\n**"NutriTrack Pro', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' Database"** — A fictional nut

In [26]:
# Simplified Text Streaming
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")

Here is a one sentence description of a fake database:

**FictaBase** is a fictional relational database containing made-up records of 10,000 imaginary customers, their fabricated purchase histories, and invented product listings for a non-existent online retailer called "ShopNever.com."

In [44]:
messages = []

prompt = """
Generate three different sample aws CLI commands. Each should be very short.
"""
append_user_message(messages, prompt)
answer = chat2(messages=messages)
print(answer)

Here are three short AWS CLI commands:

```bash
aws s3 ls

aws ec2 describe-instances

aws iam list-users
```


In [43]:
# Message Prefilling & Stop Sequences

messages = []

prompt = """
Generate three different sample aws CLI commands. Each should be very short.
"""
append_user_message(messages, prompt)

# influence llm by writing the assistant message by yourself. this message prefilling + stop sequences technique is useful
# when you want structured response, exact output structure...
append_assistant_message(messages,"here are all three commands in a single block without any commands:\n```bash")
answer = chat2(messages=messages,stop_sequences=["```"])
print(answer)


aws s3 ls

aws ec2 describe-instances

aws lambda list-functions

